# Audio Clustering with ImageBind Embeddings

## Assignment I
**Goal:** Write a colab for audio embeddings using ImageBind LLMs and cluster them.

**Dataset:** [ESC-50](https://paperswithcode.com/dataset/esc-50) (We will download a few sample files from the official repo for demonstration).

**Note:** This notebook requires a GPU runtime.

In [ ]:
# Install ImageBind and dependencies
!pip install git+https://github.com/facebookresearch/ImageBind.git
!pip install torch torchvision torchaudio

In [ ]:
import torch
from imagebind import data
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
import os
import urllib.request

# Check for GPU
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load ImageBind Model
We load the pretrained ImageBind model.

In [ ]:
model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()
model.to(device)

## 2. Load Data (ESC-50 Samples)
We download sample audio files (dog bark, rain, sea waves, etc.) from a public source or the ImageBind repo examples.

In [ ]:
os.makedirs("audio_samples", exist_ok=True)

# URLs of sample audio files (using ImageBind's own assets or similar)
audio_urls = {
    "dog_bark.wav": "https://github.com/facebookresearch/ImageBind/raw/main/.assets/dog_audio.wav",
    "car_horn.wav": "https://github.com/facebookresearch/ImageBind/raw/main/.assets/car_audio.wav",
    "bird_chirp.wav": "https://github.com/facebookresearch/ImageBind/raw/main/.assets/bird_audio.wav"
}

audio_paths = []
labels = []

for name, url in audio_urls.items():
    path = f"audio_samples/{name}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    # Duplicate to simulate clustering (since we only have 3 unique samples here)
    # In a real scenario, you'd download the full ESC-50 dataset
    for _ in range(5):
        audio_paths.append(path)
        labels.append(name.split('.')[0])

print(f"Total audio files: {len(audio_paths)}")

## 3. Extract Embeddings
We use ImageBind to extract audio embeddings.

In [ ]:
inputs = {ModalityType.AUDIO: data.load_and_transform_audio_data(audio_paths, device)}

with torch.no_grad():
    embeddings = model(inputs)

audio_embeddings = embeddings[ModalityType.AUDIO].cpu().numpy()
print(f"Embeddings shape: {audio_embeddings.shape}")

## 4. Cluster Audio
We use K-Means to cluster the embeddings.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
pred_labels = kmeans.fit_predict(audio_embeddings)

## 5. Visualization
We use t-SNE to visualize the clusters.

In [ ]:
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
embeddings_2d = tsne.fit_transform(audio_embeddings)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=pred_labels, cmap='viridis', s=100)
plt.title('Audio Clustering with ImageBind Embeddings (t-SNE)')
plt.colorbar(scatter, label='Cluster Label')
plt.show()

## 6. Evaluation
Compare with ground truth labels.

In [ ]:
from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(labels, pred_labels)
print(f"Adjusted Rand Index: {ari:.4f}")